In [0]:

from pyspark.sql import functions as F

orig = spark.table("workspace.default.freddie_orig_raw")

orig_features = orig.select(
    F.col("_c19").alias("loan_id"),
    F.lit(1).alias("orig_matched"),

    # Numeric predictors
    F.col("_c0").cast("int").alias("fico"),
    F.col("_c5").cast("double").alias("mi_pct"),
    F.col("_c6").cast("int").alias("num_units"),
    F.col("_c8").cast("double").alias("orig_cltv"),
    F.col("_c9").cast("double").alias("orig_dti"),
    F.col("_c10").cast("double").alias("orig_upb"),
    F.col("_c11").cast("double").alias("orig_ltv"),
    F.col("_c12").cast("double").alias("orig_interest_rate"),
    F.col("_c21").cast("int").alias("orig_loan_term"),
    F.col("_c22").cast("int").alias("num_borrowers"),

    # Categorical predictors
    F.col("_c2").alias("first_time_homebuyer"),
    F.col("_c7").alias("occupancy_status"),
    F.col("_c13").alias("channel"),
    F.col("_c16").alias("property_state"),
    F.col("_c17").alias("property_type"),
    F.col("_c20").alias("loan_purpose")
)

print("Loans:", orig_features.count())
print("Columns:", len(orig_features.columns))
display(orig_features.limit(10))


Loans: 50000
Columns: 18


loan_id,orig_matched,fico,mi_pct,num_units,orig_cltv,orig_dti,orig_upb,orig_ltv,orig_interest_rate,orig_loan_term,num_borrowers,first_time_homebuyer,occupancy_status,channel,property_state,property_type,loan_purpose
F17Q10000002,1,809,0.0,1,75.0,38.0,195000.0,75.0,4.25,360,1,N,P,R,PA,SF,N
F17Q10000017,1,702,0.0,1,80.0,36.0,187000.0,76.0,3.0,180,2,N,P,R,KY,SF,N
F17Q10000064,1,792,0.0,1,60.0,36.0,87000.0,60.0,3.375,360,2,N,S,R,MI,SF,N
F17Q10000065,1,776,0.0,1,80.0,18.0,106000.0,80.0,4.25,360,2,N,S,R,NY,SF,P
F17Q10000073,1,790,0.0,1,75.0,42.0,75000.0,75.0,4.25,360,1,N,I,C,UT,CO,C
F17Q10000094,1,726,0.0,1,80.0,22.0,245000.0,80.0,3.875,360,2,N,P,R,CA,SF,C
F17Q10000176,1,687,0.0,1,80.0,45.0,619000.0,75.0,3.875,360,2,N,P,R,HI,PU,C
F17Q10000300,1,766,0.0,1,72.0,27.0,185000.0,72.0,3.875,360,2,N,P,R,CO,SF,C
F17Q10000330,1,794,0.0,1,60.0,19.0,258000.0,60.0,3.375,360,1,N,P,R,HI,CO,C
F17Q10000408,1,804,0.0,1,80.0,32.0,132000.0,80.0,3.5,360,2,N,P,R,KS,SF,P


In [0]:

# Numeric missing/sentinel handling
orig_features = (
    orig_features
    .withColumn(
        "fico",
        F.when(F.col("fico").between(300, 850),
               F.col("fico"))
    )
    .withColumn(
        "orig_dti",
        F.when(F.col("orig_dti").between(0, 65),
               F.col("orig_dti"))
    )
    .withColumn(
        "orig_ltv",
        F.when(
            (F.col("orig_ltv") > 0) &
            (F.col("orig_ltv") != 999),
            F.col("orig_ltv")
        )
    )
    .withColumn(
        "orig_cltv",
        F.when(
            (F.col("orig_cltv") > 0) &
            (F.col("orig_cltv") != 999),
            F.col("orig_cltv")
        )
    )
    .withColumn(
        "mi_pct",
        F.when(F.col("mi_pct").between(0, 55),
               F.col("mi_pct"))
    )
    .withColumn(
        "num_units",
        F.when(F.col("num_units").between(1, 4),
               F.col("num_units"))
    )
    .withColumn(
        "num_borrowers",
        F.when(F.col("num_borrowers").between(1, 10),
               F.col("num_borrowers"))
    )
    .withColumn(
        "orig_upb",
        F.when(F.col("orig_upb") > 0,
               F.col("orig_upb"))
    )
)

# Categorical missing codes
unknown_codes = {
    "first_time_homebuyer": ["9"],
    "occupancy_status": ["9"],
    "channel": ["9"],
    "property_type": ["99"],
    "loan_purpose": ["9"]
}

for col_name, codes in unknown_codes.items():
    orig_features = orig_features.withColumn(
        col_name,
        F.when(
            F.col(col_name).isNull() |
            (F.trim(F.col(col_name)) == "") |
            F.col(col_name).isin(*codes),
            F.lit("Unknown")
        ).otherwise(F.col(col_name))
    )


In [0]:

orig_features = (
    orig_features
    .withColumn(
        "log_orig_upb",
        F.log("orig_upb")
    )
    .withColumn(
        "high_ltv",
        F.when(
            F.col("orig_ltv").isNull(), None
        ).when(
            F.col("orig_ltv") >= 80, 1
        ).otherwise(0)
    )
    .withColumn(
        "cltv_ltv_gap",
        F.col("orig_cltv") - F.col("orig_ltv")
    )
)


In [0]:

survival = spark.table(
    "workspace.default.freddie_survival"
)

model_base = survival.join(
    orig_features,
    on="loan_id",
    how="left"
)

# Verify join
display(
    model_base.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("loan_id").alias("unique_loans"),
        F.sum("event").alias("events"),
        F.sum(
            F.col("orig_matched").isNull().cast("int")
        ).alias("unmatched_origination")
    )
)


total_rows,unique_loans,events,unmatched_origination
49968,49968,2927,0


In [0]:

numeric_features = [
    "fico", "orig_dti", "orig_ltv",
    "orig_cltv", "orig_upb",
    "orig_interest_rate", "mi_pct",
    "num_borrowers", "log_orig_upb",
    "cltv_ltv_gap"
]

n = model_base.count()

missing_counts = model_base.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in numeric_features
]).first().asDict()

missing_summary = spark.createDataFrame(
    [
        (c, int(missing_counts[c]),
         round(100 * missing_counts[c] / n, 3))
        for c in numeric_features
    ],
    ["feature", "missing_count", "missing_pct"]
)

display(
    missing_summary.orderBy(F.desc("missing_pct"))
)

# Numeric feature distributions
display(
    model_base.select(
        "fico", "orig_dti", "orig_ltv",
        "orig_cltv", "orig_upb",
        "orig_interest_rate"
    ).summary(
        "count", "mean", "stddev",
        "min", "25%", "50%", "75%", "max"
    )
)


feature,missing_count,missing_pct
orig_dti,1806,3.614
fico,8,0.016
orig_ltv,0,0.0
orig_cltv,0,0.0
orig_upb,0,0.0
orig_interest_rate,0,0.0
mi_pct,0,0.0
num_borrowers,0,0.0
log_orig_upb,0,0.0
cltv_ltv_gap,0,0.0


summary,fico,orig_dti,orig_ltv,orig_cltv,orig_upb,orig_interest_rate
count,49960,48162,49968,49968,49968,49968
mean,744.7147918334667,35.14559196046676,74.98527057316683,75.444704610951,230090.7981107909,4.192143952129252
stddev,47.36723274862147,9.293371363296309,17.03018382487896,16.984400327381547,120636.72950852811,0.44184489644365654
min,484,1.0,6.0,6.0,11000.0,2.375
25%,711,29.0,67.0,68.0,137000.0,3.99
50%,752,36.0,80.0,80.0,207000.0,4.23
75%,784,43.0,87.0,89.0,304000.0,4.5
max,834,50.0,194.0,194.0,1000000.0,6.125


In [0]:



# HARP indicator from origination data
harp_info = orig.select(
    F.col("_c19").alias("loan_id"),
    F.trim(F.col("_c27")).alias(
        "relief_refinance_flag"
    )
)

# Join with modeling features
model_base = model_base.join(
    harp_info, on="loan_id", how="left"
)

# Missing DTI and high LTV by HARP status
display(
    model_base
    .groupBy("relief_refinance_flag")
    .agg(
        F.count("*").alias("n_loans"),
        F.sum(
            F.col("orig_dti").isNull().cast("int")
        ).alias("missing_dti"),
        F.round(
            100 * F.avg(
                F.col("orig_dti").isNull().cast("double")
            ), 2
        ).alias("missing_dti_pct"),
        F.sum(
            (F.col("orig_ltv") > 105).cast("int")
        ).alias("ltv_above_105"),
        F.max("orig_ltv").alias("max_ltv")
    )
    .orderBy("relief_refinance_flag")
)


relief_refinance_flag,n_loans,missing_dti,missing_dti_pct,ltv_above_105,max_ltv
N,48164,2,0.0,0,100.0
Y,1804,1804,100.0,90,194.0


In [0]:

# Final feature quality checks
display(
    model_base.agg(
        F.sum(
            (F.col("orig_cltv") < F.col("orig_ltv"))
            .cast("int")
        ).alias("cltv_below_ltv"),

        F.sum(
            (F.col("orig_ltv") > 105)
            .cast("int")
        ).alias("high_ltv_loans"),

        F.sum(
            (
                (F.col("relief_refinance_flag") == "Y") &
                (F.col("orig_ltv") > 80)
            ).cast("int")
        ).alias("harp_loans"),

        F.sum(
            F.col("relief_refinance_flag").isNull()
            .cast("int")
        ).alias("missing_relief_flag")
    )
)


cltv_below_ltv,high_ltv_loans,harp_loans,missing_relief_flag
0,90,433,0


In [0]:

model_base.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.freddie_model_base"
    )

print("Modeling feature table saved!")


Modeling feature table saved!
